# Post-Training Explorer: Full Training Pipeline
**SNIA DSN AI Stack Webinar Series — 2026**

This notebook runs the complete post-training pipeline:
1. **SFT** (Supervised Fine-Tuning) — ~12 min on T4
2. **DPO** (Direct Preference Optimization) — ~8 min on T4
3. **GRPO** (Group Relative Policy Optimization) — ~35 min on T4
4. **Export** — Generates `precomputed_results.json` for the web app
5. **ONNX Conversion** — Converts models for browser-side inference

**GPU Required:** Go to Runtime → Change runtime type → T4 GPU

Each section can be run independently. If you hit a Colab timeout during GRPO, run SFT+DPO in one session and GRPO in a second session.

In [ ]:
# ============================================================
# Setup: Install dependencies and prepare scripts
# ============================================================

# Install all required packages
!pip install -q torch transformers datasets accelerate peft trl sentencepiece safetensors

# Option 1: Clone from GitHub (recommended — already pushed)
!git clone https://github.com/provandal/post-training-explorer.git /content/FineTuningDemo

# Option 2: If you uploaded the project as a zip instead, uncomment this:
# !unzip -q /content/FineTuningDemo.zip -d /content/FineTuningDemo

# --- Set working directory ---
import os
PROJECT_ROOT = "/content/FineTuningDemo"
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")
print(f"Scripts present: {os.path.exists('scripts/train_sft.py')}")

# List available scripts
if os.path.exists('scripts'):
    for f in sorted(os.listdir('scripts')):
        print(f"  scripts/{f}")
else:
    print("WARNING: scripts/ directory not found!")
    print("Upload your scripts/ folder or adjust PROJECT_ROOT above.")

In [ ]:
# ============================================================
# Verify GPU availability
# ============================================================

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"Memory: {mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected!")
    print("Go to Runtime → Change runtime type → T4 GPU")
    print("Training will be extremely slow on CPU.")

---
## Step 1: Supervised Fine-Tuning (SFT)

SFT is the foundation step. We teach SmolLM2-360M **what** to output — correct storage I/O workload classifications with concise reasoning — using LoRA adapters.

**What happens:**
- Generates 480 synthetic training examples (80 per workload class)
- Captures base model outputs **before** training (for comparison)
- Trains LoRA adapters (rank=16) on attention projections — only ~0.5% of parameters
- Captures fine-tuned outputs **after** training
- Saves adapter weights, training loss curve, and before/after comparisons

**Expected runtime:** ~12 minutes on T4 GPU  
**Output directory:** `scripts/outputs/sft/`

In [ ]:
# ============================================================
# Run SFT training
# ============================================================
# This trains the base SmolLM2-360M model to classify storage
# I/O workloads using LoRA adapters. ~12 min on T4.
#
# Outputs:
#   scripts/outputs/sft/adapter/         - LoRA adapter weights
#   scripts/outputs/sft/training_loss.json - Loss curve data
#   scripts/outputs/sft/lora_weights.json  - LoRA weight visualization
#   scripts/outputs/sft/before_after_comparison.json
# ============================================================

!python scripts/train_sft.py

---
## Step 2: Direct Preference Optimization (DPO)

DPO refines the SFT model's **style** — teaching it to prefer concise, confident responses over verbose, hedging ones. It works by training on (chosen, rejected) pairs without needing a separate reward model.

**What happens:**
- Loads the SFT adapter from Step 1 and merges it into the base model
- Generates 300 preference pairs (50 per class) with three rejection strategies:
  - Wrong classification label
  - Correct label but excessively verbose/hedging
  - Wrong label with hedging
- Trains a new LoRA adapter on the DPO objective
- Computes chosen/rejected log-probability shifts to show how DPO changed the model

**Prerequisite:** Step 1 (SFT) must complete first  
**Expected runtime:** ~8 minutes on T4 GPU  
**Output directory:** `scripts/outputs/dpo/`

In [ ]:
# ============================================================
# Run DPO training
# ============================================================
# Requires the SFT adapter from Step 1.
# Trains the model to prefer concise, correct responses. ~8 min on T4.
#
# Outputs:
#   scripts/outputs/dpo/adapter/              - DPO LoRA adapter
#   scripts/outputs/dpo/training_curves.json   - Loss + reward margins
#   scripts/outputs/dpo/probability_shifts.json - Chosen/rejected log-prob shifts
#   scripts/outputs/dpo/preference_pair_examples.json
# ============================================================

!python scripts/train_dpo.py

---
## Step 3: Group Relative Policy Optimization (GRPO)

GRPO is an **online RL** approach — unlike DPO which uses pre-generated pairs, GRPO generates completions on-the-fly and scores them with a reward function.

**What happens:**
- Loads the SFT adapter and merges it (starts from SFT, not DPO)
- For each prompt, generates K=8 completions using sampling
- Scores each completion with a binary reward function (correct classification = 1.0)
- Computes group-relative advantages: `(reward - group_mean) / group_std`
- Updates the policy with REINFORCE-style gradients + KL penalty

**Prerequisite:** Step 1 (SFT) must complete first (DPO is not required)  
**Expected runtime:** ~35 minutes on T4 GPU  
**Output directory:** `scripts/outputs/grpo/`

> **Timeout warning:** If Colab disconnects during GRPO, your SFT and DPO adapters
> are already saved. Reconnect, re-run the Setup cell, then run GRPO in a new session.
> The script checks for the SFT adapter automatically.

In [ ]:
# ============================================================
# Run GRPO training
# ============================================================
# Requires the SFT adapter from Step 1.
# Uses online RL with a binary reward function. ~35 min on T4.
#
# If TRL >= 0.12 is installed, uses TRL's GRPOTrainer.
# Otherwise falls back to a simplified manual implementation.
#
# Outputs:
#   scripts/outputs/grpo/adapter/             - GRPO LoRA adapter
#   scripts/outputs/grpo/training_curves.json  - Loss + reward curves
#   scripts/outputs/grpo/generation_logs.json   - Sample generations with scores
#   scripts/outputs/grpo/group_statistics.json  - Accuracy/reward over training
# ============================================================

!python scripts/train_grpo.py

---
## Step 4: Export Artifacts

This step loads every model variant (base, SFT, DPO, GRPO) and runs inference on a standardized set of 20 test prompts. The output is a single `precomputed_results.json` file that the web app uses to display real model behavior.

**What happens:**
- Builds 20 deterministic test prompts (balanced across 6 workload categories)
- Loads each available model variant and runs greedy inference
- Captures generated text, token probabilities, and timing
- Collects all training artifacts (loss curves, LoRA weights, preference pairs, etc.)
- Saves everything into one consolidated JSON file

**Prerequisite:** At least Step 1 (SFT) must be complete. DPO and GRPO are optional — the script exports whatever models are available.  
**Expected runtime:** ~3 minutes on T4 GPU  
**Output:** `app/public/data/precomputed_results.json`

In [ ]:
# ============================================================
# Run export
# ============================================================
# Evaluates all available model variants on 20 test prompts
# and produces precomputed_results.json for the web app.
#
# The script will skip any missing adapters and export
# results for whatever models are available.
#
# Outputs:
#   app/public/data/precomputed_results.json  - Main results file
#   app/public/data/resource_summary.json     - Resource utilization
#   scripts/outputs/export/                   - Backup copies
# ============================================================

!python scripts/export_artifacts.py

---
## Step 5: Convert to ONNX for Browser Inference

This step converts two model variants to ONNX format and pushes them to HuggingFace Hub, enabling **client-side inference** directly in the browser via `@huggingface/transformers` (transformers.js).

**Two models are converted:**
1. **Base model** (untrained SmolLM2-360M) — shows the model's behavior before any training
2. **GRPO model** (SFT + GRPO merged) — shows the fully trained result

**What happens:**
- Saves the unmodified base model in a clean directory
- Merges SFT adapter → merges GRPO adapter on top → saves the merged model
- Converts both to ONNX using `optimum`
- Pushes both ONNX models to HuggingFace Hub as separate repos

**Prerequisites:** Steps 1 (SFT) and 3 (GRPO) must be complete. You must be logged in to HuggingFace Hub.
**Expected runtime:** ~5 minutes on T4 GPU
**Output:** Two HuggingFace Hub repos with ONNX models

In [ ]:
# ============================================================
# Convert models to ONNX and push to HuggingFace Hub
# ============================================================
# Requires SFT and GRPO adapters from Steps 1 and 3.
# Converts base and merged GRPO models to ONNX format,
# then pushes to HuggingFace Hub for browser-side inference.
#
# You must be logged in to HuggingFace Hub first.
# ============================================================

# Install ONNX conversion dependencies
!pip install -q optimum[onnxruntime] onnxruntime huggingface_hub

# Log in to HuggingFace Hub (will prompt for token if not already logged in)
from huggingface_hub import login
login()

# Run the conversion script
!python scripts/convert_to_onnx.py

---
## Training the 1.7B Model (Optional)

If you want to see how model size affects post-training results, you can train SmolLM2-1.7B using the same pipeline. The `--model-size 1.7B` flag adjusts batch sizes and output directories automatically.

**Warning:** The 1.7B model uses significantly more VRAM. SFT should work on T4, but GRPO may need an A100 runtime.

**Expected runtimes on T4 GPU:**
- SFT: ~55 minutes
- DPO: ~35 minutes  
- GRPO: ~150 minutes (or ~45 min on A100)

In [ ]:
# ============================================================
# Train 1.7B model (optional — run after 360M is complete)
# ============================================================
# Uncomment the lines below to train the larger model.
# The --model-size flag adjusts batch sizes and output paths.

# !python scripts/train_sft.py --model-size 1.7B
# !python scripts/train_dpo.py --model-size 1.7B
# !python scripts/train_grpo.py --model-size 1.7B
# !python scripts/export_artifacts.py --all-models

---
## AI Research Advisor (AINOS Lite)

The AI advisor analyzes your training results and recommends what experiment to try next. This demonstrates the AINOS concept: AI proposes, human approves, system executes.

**Requires:** `ANTHROPIC_API_KEY` environment variable set with a valid Claude API key.

In [ ]:
# ============================================================
# AI Research Advisor
# ============================================================
# Analyzes your training results and suggests next experiments.
# Requires an Anthropic API key.

!pip install -q anthropic rich

import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # Uncomment and paste your key

# Dry run first to see what will be sent to Claude
!python scripts/ai_advisor.py --dry-run

# Uncomment to get the actual recommendation:
# !python scripts/ai_advisor.py

---
## Validation

Verify that the exported results contain all expected model variants and check the accuracy summary. You should see:
- **base**: Low accuracy (the untrained model doesn't know the task)
- **sft**: Moderate-to-high accuracy (learned the classification format)
- **dpo**: Similar accuracy to SFT but with more concise/confident outputs
- **grpo**: Highest accuracy (optimized directly for classification correctness)

In [ ]:
# ============================================================
# Validate exported results
# ============================================================

import json
import os

results_path = 'app/public/data/precomputed_results.json'

if not os.path.exists(results_path):
    # Try the export directory backup
    results_path = 'scripts/outputs/export/precomputed_results.json'

with open(results_path) as f:
    results = json.load(f)

print(f"Models evaluated: {results['metadata']['model_variants']}")
print(f"Test prompts: {results['metadata']['num_test_prompts']}")
print(f"Categories: {results['metadata']['categories']}")
print(f"Export time: {results['metadata']['export_timestamp']}")
print()

print("Accuracy summary:")
for variant in results['metadata']['model_variants']:
    s = results['model_results'][variant]['summary']
    print(f"  {variant:>6s}: {s['accuracy']:.1%} ({s['correct']}/{s['total']}) "
          f"  avg_time={s['avg_generation_time_ms']:.1f}ms")

print()
print("Training artifacts included:")
for key in ['training_data', 'dpo_preference_examples', 'grpo_generation_logs',
            'lora_weight_visualization', 'sft_before_after',
            'dpo_probability_shifts', 'grpo_group_statistics',
            'resource_utilization']:
    status = 'yes' if key in results else 'MISSING'
    print(f"  {key}: {status}")

print()
file_size_mb = os.path.getsize(results_path) / 1e6
print(f"Results file size: {file_size_mb:.2f} MB")

In [ ]:
# ============================================================
# Download precomputed_results.json to your local machine
# ============================================================

from google.colab import files

download_path = 'app/public/data/precomputed_results.json'
if not os.path.exists(download_path):
    download_path = 'scripts/outputs/export/precomputed_results.json'

files.download(download_path)
print(f"Downloaded: {download_path}")

---
## Next Steps

1. **Copy the results file** to your local project:
   ```
   app/public/data/precomputed_results.json
   ```

2. **Update model IDs** in `app/src/services/inference.js` with the repo names printed by the ONNX conversion script:
   ```javascript
   const MODEL_CONFIGS = {
     base: { id: '<your-username>/smollm2-360m-storage-io-base-onnx', ... },
     grpo: { id: '<your-username>/smollm2-360m-storage-io-grpo-onnx', ... },
   }
   ```

3. **Update Epilogue links** in `app/src/stops/Epilogue.jsx` with your real HuggingFace/Colab URLs.

4. **Build the web app** to verify it picks up the real data:
   ```bash
   cd app
   npm install
   npm run build
   ```

5. **Run locally** to preview:
   ```bash
   npm run dev
   ```

The web app will automatically detect and use `precomputed_results.json` — no code changes needed for the tour and explore views. The LiveInferencePanel will download ONNX models from HuggingFace Hub for client-side browser inference.